<a href="https://colab.research.google.com/github/Sk-amit26/PPD-PROJECT/blob/main/PPDII.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Fri Aug 14 05:13:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q kagglehub tensorflow pillow numpy


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving model.py to model (2).py
Saving losses_metrics.py to losses_metrics (2).py
Saving train.py to train (2).py
Saving requirements.txt to requirements (2).txt
Saving data_loader.py to data_loader (2).py
Saving README.md to README (2).md


In [ ]:
!ls -la mscp_best.weights.h5

-rw-r--r-- 1 root root 33109840 Aug 14 05:54 mscp_best.weights.h5


In [ ]:
import kagglehub

path = kagglehub.dataset_download("maamri95/cdnet2014")
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/maamri95/cdnet2014/versions/1


In [ ]:
import os
if not os.path.isdir("/content/cdnet2014_local/dataset"):
    !cp -r {path} /content/cdnet2014_local
CDNET_ROOT = "/content/cdnet2014_local/dataset"
print(os.listdir(CDNET_ROOT))

In [ ]:
import os
CDNET_ROOT = "/content/cdnet2014_local/dataset"
print(os.listdir(CDNET_ROOT))

['lowFramerate', 'badWeather', 'nightVideos', 'dynamicBackground', 'cameraJitter', 'baseline', 'shadow', 'intermittentObjectMotion', 'PTZ', 'turbulence', 'thermal']


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Memory growth enabled on {len(gpus)} GPU(s)")

Memory growth enabled on 1 GPU(s)


In [ ]:
from data_loader import collect_dataset, sample_training_frames

pairs = collect_dataset(CDNET_ROOT, categories=("thermal",))
print(f"Found {len(pairs)} thermal frame pairs")

Found 18055 thermal frame pairs


In [ ]:
!ls -la mscp_best.weights.h5

-rw-r--r-- 1 root root 33109840 Aug 14 05:54 mscp_best.weights.h5


In [ ]:
from model import build_mscp_model

target_size = (240, 320)
model = build_mscp_model(input_shape=(240, 320, 3))
model.load_weights("mscp_best.weights.h5")
print("Weights loaded.")

Weights loaded.


In [ ]:
import numpy as np
import time
from data_loader import load_frame
from losses_metrics import ConfusionMatrixMetrics

def evaluate_with_progress(model, pairs, target_size, threshold=0.9, batch_size=4):
    metrics = ConfusionMatrixMetrics(threshold=threshold)
    n = len(pairs)
    start = time.time()

    for i in range(0, n, batch_size):
        batch = pairs[i:i+batch_size]
        imgs, labels, masks = [], [], []
        for in_path, gt_path in batch:
            img, label, valid_mask = load_frame(in_path, gt_path, target_size)
            imgs.append(img)
            labels.append(label)
            masks.append(valid_mask)
        imgs = np.stack(imgs)
        preds = model(imgs, training=False).numpy()
        for j in range(len(batch)):
            metrics.update(labels[j], preds[j], masks[j])

        if (i // batch_size) % 100 == 0:
            elapsed = time.time() - start
            pct = 100 * min(i + batch_size, n) / n
            print(f"  {pct:5.1f}%  ({i+batch_size}/{n} frames)  elapsed={elapsed:.0f}s")

    print(f"Done in {time.time()-start:.0f}s")
    return metrics.report()

In [ ]:
results = evaluate_with_progress(model, pairs, target_size, threshold=0.9, batch_size=4)
for k, v in results.items():
    print(f"{k}: {v:.4f}")

    0.0%  (4/18055 frames)  elapsed=0s
    2.2%  (404/18055 frames)  elapsed=26s
    4.5%  (804/18055 frames)  elapsed=51s
    6.7%  (1204/18055 frames)  elapsed=77s
    8.9%  (1604/18055 frames)  elapsed=102s
   11.1%  (2004/18055 frames)  elapsed=127s
   13.3%  (2404/18055 frames)  elapsed=153s
   15.5%  (2804/18055 frames)  elapsed=179s
   17.7%  (3204/18055 frames)  elapsed=204s
   20.0%  (3604/18055 frames)  elapsed=229s
   22.2%  (4004/18055 frames)  elapsed=255s
   24.4%  (4404/18055 frames)  elapsed=280s
   26.6%  (4804/18055 frames)  elapsed=305s
   28.8%  (5204/18055 frames)  elapsed=330s
   31.0%  (5604/18055 frames)  elapsed=355s
   33.3%  (6004/18055 frames)  elapsed=381s
   35.5%  (6404/18055 frames)  elapsed=406s
   37.7%  (6804/18055 frames)  elapsed=431s
   39.9%  (7204/18055 frames)  elapsed=456s
   42.1%  (7604/18055 frames)  elapsed=481s
   44.3%  (8004/18055 frames)  elapsed=506s
   46.5%  (8404/18055 frames)  elapsed=532s
   48.8%  (8804/18055 frames)  elapsed=557

In [ ]:
print(results)

{'Precision': 0.9865859661723724, 'Recall': 0.933461165692636, 'F-measure': 0.9592886244877933, 'Specificity': 0.9989705095668232, 'PWC': 0.5944596806610566}
